In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_file_date","2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
from pyspark.sql import functions as F

movies_df = spark.read.table("movie_silver.movies") \
                        .filter(f"file_date = '{v_file_date}'")  
languages_df = spark.read.table("movie_silver.languages")
movies_languages_df = spark.read.table("movie_silver.movies_languages") \
                        .filter(f"file_date = '{v_file_date}'")  
genres_df = spark.read.table("movie_silver.genres")
movies_genres_df = spark.read.table("movie_silver.movies_genres") \
                        .filter(f"file_date = '{v_file_date}'")  

In [0]:
result_df = (
    movies_df.alias("m")
    .filter(F.col("year_release_date") >= 2000)
    .join(
        movies_languages_df.alias("ml"),
        F.col("m.movie_id") == F.col("ml.movie_id"),
        "inner"
    )
    .join(
        languages_df.alias("l"),
        F.col("ml.language_id") == F.col("l.language_id"),
        "inner"
    )
    .join(
        movies_genres_df.alias("mg"),
        F.col("m.movie_id") == F.col("mg.movie_id"),
        "inner"
    )
    .join(
        genres_df.alias("g"),
        F.col("mg.genre_id") == F.col("g.genre_id"),
        "inner"
    )
    .select(
        F.col("m.movie_id"),
        F.col("l.language_id"),
        F.col("g.genre_id"),
        F.col("m.title"),
        F.col("m.duration_time"),
        F.col("m.release_date"),
        F.col("m.vote_average"),
        F.col("l.language_name"),
        F.col("g.genre_name")
    )
    .withColumn("created_date", F.lit(v_file_date))
    .orderBy(F.desc("release_date"))
)

In [0]:
#overwrite_partition("movie_gold","results_movie_genre_language","created_date", v_file_date)

In [0]:
merge_condition = "tgt.movie_id = src.movie_id AND tgt.language_id = src.language_id AND tgt.genre_id = src.genre_id AND tgt.created_date = src.created_date"
merge_delta_lake(result_df,"movie_gold","results_movie_genre_language",merge_condition,"created_date")

In [0]:
%sql
SELECT created_date,
COUNT(1)
FROM movie_gold.results_movie_genre_language
GROUP BY created_date

created_date,count(1)
2024-12-16,5494
2024-12-23,1024
2024-12-30,897
